<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Linear probes and few-shot

**[Linear probes and few-shot](https://learning.nextia-ai.com/courses/vision/m03/linear-probes-and-few-shot/)** · Vision AI: See, Count and Inspect · Module 3, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** train a logistic regression on stored embeddings (a linear probe), fix a class with few photos with balanced class weights, and draw a learning curve with 1 to all labelled photos per class.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No model download, no account and no key. |
| **You should know** | Embeddings, from [Image embeddings](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/), and zero-shot, from [CLIP zero-shot](https://learning.nextia-ai.com/courses/vision/m03/clip-zero-shot/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m03/linear-probes-and-few-shot/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M03-L03-linear-probes-and-few-shot/linear-probes-and-few-shot-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data and the probe

The course team stored the embeddings of every catalogue photo for three models: CLIP ViT-B/32, DINOv2 small and ResNet-18 (`data/embeddings/`). A **linear probe** is a logistic regression on these numbers. The model that made them does not change.

In [ ]:
from sklearn.linear_model import LogisticRegression
catalogue = pd.read_csv("data/catalogue/manifest.csv")
train = catalogue[catalogue["split"] == "train"].reset_index(drop=True)
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)
y_train, y_test = train["label"].values, list(test["label"])

def embeddings(kind, ids):
    """The course's stored embeddings (data/embeddings/<kind>.npz), scaled to length 1."""
    z = np.load(f"data/embeddings/{kind}.npz")
    X = np.stack([z[i] for i in ids]).astype(np.float32)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def probe(kind, rows=None, balanced=False):
    """A logistic regression on the embeddings of the chosen training rows; the predictions for the test photos."""
    rows = np.arange(len(train)) if rows is None else rows
    model = LogisticRegression(C=2.0, max_iter=5000, class_weight="balanced" if balanced else None)
    model.fit(embeddings(kind, train["image_id"][rows]), y_train[rows])
    return model.predict(embeddings(kind, test["image_id"]))

def draw(n, seed):
    """n training rows of each class (all of them if the class has fewer), chosen at random."""
    rng = np.random.default_rng(seed)
    return np.concatenate([rng.permutation(np.where(y_train == k)[0])[:n] for k in CLASSES])
print(len(train), "training photos;", pd.Series(y_train).value_counts()["candle"], "of them are candles")

> **Check.** You should see `453 training photos; 15 of them are candles`.

## 2. A probe on all the labels

> **Predict.** Candle has only 15 training photos, and the other classes 41 to 70. Will candle be the best or the worst class?

In [ ]:
scores = cp.classification_scores(y_test, list(probe("clip")), CLASSES)
print("accuracy", scores["accuracy"])
pd.DataFrame(scores["per_class"]).T[["precision", "recall"]]

> **Check.** Accuracy 0.894. Candle has a recall of 0.6: the probe finds only 12 of the 20 test candles. Its precision is 1.0: when it says candle, it is right.

## 3. Balanced classes

`class_weight="balanced"` gives each class the same total weight in training: each candle photo counts about four times as much as a chair photo.

In [ ]:
balanced = cp.classification_scores(y_test, list(probe("clip", balanced=True)), CLASSES)
print("accuracy", balanced["accuracy"], " candle recall", balanced["per_class"]["candle"]["recall"])

> **Check.** Accuracy 0.944, candle recall 0.95.

## 4. A learning curve

How many labelled photos per class does the probe need? The next cell trains it with 1, 5, 10, 20, 40 and 80 photos per class (80 means all of them), 10 times with a different random draw, and takes the mean. It takes about 30 seconds.

> **Predict.** Will the accuracy rise all the way to "all labels"?

In [ ]:
def curve(kind, sizes=(1, 5, 10, 20, 40, 80), draws=10):
    out = {}
    for n in sizes:
        accs = [cp.classification_scores(y_test, list(probe(kind, draw(n, seed))), CLASSES)["accuracy"]
                for seed in range(draws)]
        out[n] = round(float(np.mean(accs)), 3)
    return out

clip_curve = curve("clip")
print(clip_curve)
plt.plot(list(map(str, clip_curve)), list(clip_curve.values()), marker="o", label="CLIP probe")
plt.axhline(0.925, color="grey", linestyle="--", label="CLIP zero-shot, eight templates (no labels)")
plt.ylim(0.5, 1); plt.xlabel("labelled photos per class (80 = all)"); plt.ylabel("test accuracy"); plt.legend(); plt.show()

> **Check.** The curve rises from 0.624 (1 photo per class) to 0.928 (20 per class), then falls to 0.894 with all labels. More photos make the classes unbalanced: candle stays at 15 while the others grow.

## 5. Your turn: DINOv2 with 20 photos per class

> **Your turn.** Use the function `curve` with the DINOv2 embeddings (`"dinov2"`) and only the size 20. Put the mean accuracy in `dinov2_20`.

In [ ]:
dinov2_20 = curve("dinov2", sizes=(20,))[20]
print(dinov2_20)

In [ ]:
expect('dinov2_20', dinov2_20, 0.893, tolerance=0.0005)

## 6. The comparison table

Your probes next to the zero-shot and transfer-learning rows that the course team measured on the same test photos:

In [ ]:
table = cp.Table.load("data/comparison_table.json")
table.show("catalogue", columns=("method", "setting", "value", "worst", "worst_value", "labelled_images"))

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m03/linear-probes-and-few-shot/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Labels: how many, and how good](https://learning.nextia-ai.com/courses/vision/m03/labels-how-many-and-how-good/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m03/linear-probes-and-few-shot/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.